In [1137]:
import torch

In [1138]:
def sample_sphere(n):
    """Sample a point on a sphere"""
    z = 2*torch.rand(n) - 1
    phi = 2*(torch.pi)*torch.rand(n)
    r = torch.sqrt(1-z**2)
    x, y = r*torch.cos(phi), r*torch.sin(phi)
    return torch.concat([x.reshape(-1,1),y.reshape(-1,1),z.reshape(-1,1)], axis=1)
    

In [1139]:
coord = sample_sphere(4)

In [1140]:
coord

tensor([[ 0.3913, -0.9169, -0.0788],
        [ 0.8245,  0.4908,  0.2816],
        [-0.7942,  0.4530,  0.4050],
        [-0.7232,  0.6330,  0.2761]])

In [1141]:
torch.square(coord)

tensor([[0.1531, 0.8407, 0.0062],
        [0.6798, 0.2409, 0.0793],
        [0.6307, 0.2053, 0.1640],
        [0.5231, 0.4007, 0.0762]])

In [1142]:
def check_validity(coord):
    assert  torch.all(torch.abs(torch.sum((torch.square(coord)), axis=1)  - 1) <= 1e-3)

In [1143]:
check_validity(coord)

In [1144]:
x,y = sample_sphere(4), sample_sphere(4)

In [1145]:
x.shape, y.shape

(torch.Size([4, 3]), torch.Size([4, 3]))

In [1146]:

def log_map(x,y):
    """Finds the minimum geodesic or minor arc for circle"""
    assert x.shape==y.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],y[i]) for i in range(n)]).reshape(-1,1)
    theta = torch.tensor([torch.arccos(dot) for dot in dot_prod]).reshape(-1,1)
    y_parallel = dot_prod * x
    y_perp = y - y_parallel
    y_perp_norm = torch.nn.functional.normalize(y_perp, 2, dim=-1)
    v = theta * y_perp_norm
    return v


        

In [1147]:
v = log_map(x,y)

In [1148]:
v.shape

torch.Size([4, 3])

In [1149]:
def test_tangentness(x, v, tol=1e-5):
    assert x.shape==v.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],v[i]) for i in range(n)])
    tol_check = torch.abs(dot_prod)<=tol
    assert torch.all(tol_check)


In [1150]:
test_tangentness(x,v)

In [1151]:
def log_map_length_test(x,y,v, tol=1e-5):
    assert x.shape == y.shape
    assert x.shape == v.shape
    n = x.shape[0]
    theta = torch.tensor([torch.arccos(torch.dot(x[i], y[i])) for i in range(n)])
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1)
    assert torch.all(torch.abs(v_norm - theta) <= tol)


In [1152]:
log_map_length_test(x,y,v)

In [1153]:
def exp_map(x,v):
    assert x.shape==v.shape
    n = x.shape[0]
    v_norm = torch.nn.functional.normalize(v, p=2, dim=-1)
    theta = torch.linalg.vector_norm(v, ord=2, dim=-1)
    y = torch.cos(theta).reshape(-1,1) * x + torch.sin(theta).reshape(-1,1) * v_norm
    return y
    

In [1154]:
exp_map(x,v)

tensor([[ 0.1223, -0.9802,  0.1556],
        [-0.1000,  0.0231,  0.9947],
        [-0.1092, -0.9936, -0.0282],
        [ 0.1348,  0.8415,  0.5231]])

In [1155]:
def test_exp(x,y, tol=1e-5):
    assert torch.all(torch.abs(exp_map(x, log_map(x,y)) - y) <= tol)

In [1156]:
test_exp(x,y)

In [1157]:
def test_unit_norm(x,v, tol=1e-5):
    assert torch.all(torch.abs(torch.linalg.vector_norm(exp_map(x,v), ord=2, dim=-1)-1) <= tol)

In [1158]:
test_unit_norm(x,v)

In [1159]:
def premetric_d(x,y):
    assert x.shape == y.shape
    n = x.shape[0]
    return torch.tensor(
        [torch.arccos(torch.clamp(torch.dot(x[i], y[i]), min=-1, max=1)) for i in range(n)]
        )

In [1160]:
premetric_d(x,y)

tensor([0.3239, 2.2560, 1.5488, 1.2384])

In [1161]:
def test_d_x_x_zero(x,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,x) - 0
        ) <= tol
    )

In [1162]:
test_d_x_x_zero(x)

In [1163]:
def test_d_symmetry(x,y,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,y) - premetric_d(y,x)
        ) <= tol
    )

In [1164]:
test_d_symmetry(x,y)

In [1165]:
def test_d_non_neg(x,y):
    assert torch.all(
        premetric_d(x,y) >= 0
    )

In [1166]:
test_d_non_neg(x,y)

In [1167]:
def grad_d(x, y):
    v = log_map(x,y)
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1).reshape(-1,1)
    return -v * torch.reciprocal(v_norm)

In [1168]:
g = grad_d(x,y)

In [1169]:
test_tangentness(x,g)

In [1170]:
test_unit_norm(x,g)

In [1171]:
def test_against_log_map(x,y, tol=1e-3):
    assert torch.all(torch.abs(
        torch.nn.functional.normalize(log_map(x,y),p=2,dim=-1) + grad_d(x,y)
    ) <= tol
    )

In [1172]:
test_against_log_map(x,y)

In [1173]:
def get_time_sched_and_derivative(t):
    assert torch.all(t < 1)
    assert torch.all(t>=0)
    return 1-t, -1*torch.reciprocal(1-t)

In [1174]:
get_time_sched_and_derivative(torch.tensor([0,0.5]))

(tensor([1.0000, 0.5000]), tensor([-1., -2.]))

In [1175]:
def conditional_vf(x, x1, t):
    assert x.shape == x1.shape
    assert x.shape[0] == t.shape[0]
    grad = grad_d(x,x1)
    pre = premetric_d(x,x1).reshape(-1,1)
    _, log_deriv = get_time_sched_and_derivative(t)
    log_deriv = log_deriv.reshape(-1,1)
    return log_deriv * pre * grad * \
            torch.reciprocal(
                torch.square(
                torch.linalg.vector_norm(grad,ord=2,dim=-1)
                ).reshape(-1,1)
                )

    


In [1176]:
def get_time_samples(n):
    return torch.rand(n)

In [1177]:
t = get_time_samples(4)

In [1178]:
u = conditional_vf(x,y,t)

In [1179]:
test_tangentness(x,u)

In [1180]:
t.shape

torch.Size([4])

In [1181]:
def geodesic_path(x0, x1, t):
    return exp_map(x0, t.reshape(-1,1)*log_map(x0,x1))

In [1182]:
geodesic_path(x,y,t)

tensor([[-0.0302, -0.9167,  0.3985],
        [ 0.9867,  0.0993, -0.1285],
        [-0.1019, -0.9946,  0.0205],
        [ 0.5923,  0.6295,  0.5029]])

In [1183]:
def test_start(x, y, tol=1e-3):
    assert x.shape == y.shape
    n = x.shape[0]
    t = torch.zeros(n,1)
    assert torch.all(torch.abs(geodesic_path(x,y,t) - x)<=tol)


In [1184]:
test_start(x,y)

In [1185]:
def test_end(x, y, tol=1e-3):
    assert x.shape == y.shape
    n = x.shape[0]
    t = torch.ones(n,1)
    assert torch.all(torch.abs(geodesic_path(x,y,t) - y)<=tol)

In [1186]:
test_end(x,y)

In [1187]:
def test_distance_schedule(x0, x1, t, tol=1e-3):
    t = t.reshape(-1, 1)
    xt = exp_map(x0, t * log_map(x0, x1))
    assert torch.all(
        torch.abs(
            premetric_d(xt, x1).reshape(-1,1) - ((1-t) * premetric_d(x0, x1).reshape(-1,1))
        ) <= tol
        
    )

In [1188]:
test_distance_schedule(x,y,t)

In [1189]:
t_sweep = torch.tensor([0, 0.25, 0.5, 0.75, 1])
n=4
t_sweep.repeat(n,1)

tensor([[0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000]])

In [1190]:
def test_sweep(x0, x1):
    assert x0.shape == x1.shape
    n = x0.shape[0]
    t_sweep = torch.tensor([0, 0.25, 0.5, 0.75, 1]).repeat(n, 1)
    n_t = t_sweep.shape[1]
    for i in range(n_t):
        test_distance_schedule(x0, x1, t_sweep[:, i])
       



In [1191]:
test_sweep(x,y)